# IsolationForest Health Score — CED Machine Test Notebook

Tests the full pipeline: config → DataHandler → Model.train() → Model.real_time_inference()

## Known bugs fixed in this notebook
1. **`__repr__` bug in `Model`** — returns `None` instead of a string (print vs return)
2. **`timestamp` included in feature matrix** — config's `required_features` list contains `'timestamp'`, which leaks into `X` if not dropped carefully; `DataHandler.fetch_train_data` drops it, but the notebook's manual `X.reshape` path using `training_data.csv` did not
3. **`handler.df` assigned a flat CSV** — after `fetch_train_data()` the notebook re-assigned `handler.df = df` (the flat 670-row CSV). This is fine for re-use but the column set must match what `fetch_train_data` expects (no `target` column, timestamp present)
4. **`real_time_inference` with isolation_forest** — the method is in `ROWWISE_MODELS`, so the `elif ROWWISE_MODELS` branch in `real_time_inference` skips the `reshape(1,-1)` and keeps `X` as `(seq_len, F)`. For `history_window=1` this is `(1, 72)` which is correct, but the branch check uses `not in ROWWISE_MODELS` — isolation_forest IS in ROWWISE_MODELS, so it skips both reshapes and passes raw `(1, 72)` to `predict()` — **this is actually correct** for sklearn row-wise, but only because seq_len=1
5. **`model.train()` passes `X` shaped `(N, 1, 72)` → reshapes to `(N, 72)`** correctly via UNSUPERVISED_MODELS branch (isolation_forest is in UNSUPERVISED_MODELS). This is correct.


## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # non-interactive backend; switch to 'inline' in Jupyter
%matplotlib inline

from data_handler import DataHandler
from model import Model

## 1. Load config

In [ ]:
with open("../config/analysis_config.yaml") as f:
    cfg = yaml.safe_load(f)

method_config = cfg["target"]["ced_machine__health_score"][0]

print("method   :", method_config["method"])
print("model_type:", method_config["model_type"])
print("save_name :", method_config["save_name"])
print("arch      :", method_config["arch"])
print("n features:", len(method_config["required_features"]))

## 2. Load raw CED data and build training windows via DataHandler

In [ ]:
raw_df = pd.read_csv('../data/2602_to_0903.csv')  # adjust path as needed
print(f"Raw data shape: {raw_df.shape}")
print(raw_df.dtypes[:5])

In [ ]:
# Init DataHandler
handler = DataHandler(config=method_config, target_name="ced_machine__health_score")

# Ingest rows (long-format: timestamp, station_name, metric_name, value)
rows = raw_df[["timestamp", "station_name", "metric_name", "value"]].to_records(index=False)
for ts, st, mt, val in rows:
    handler._process_one(f"{st}__{mt}", val, ts)

# Build training windows
X, y, timestamps = handler.fetch_train_data()

print(f"\nX shape     : {X.shape}   (N windows, seq_len=1, n_features)")
print(f"y           : {y}  — expected None for unsupervised")
print(f"timestamps  : {timestamps.shape if timestamps is not None else None}")

## 2b. Alternative: load pre-built training_data.csv (skip raw ingest)

In [ ]:
# ─── Use this cell instead of section 2 if you already have training_data.csv ───
flat_df = pd.read_csv('training_data.csv')
print(f"Loaded flat training data: {flat_df.shape}")

# Reconstruct X (N, 1, n_features) from the flat CSV
feature_cols = [c for c in flat_df.columns if c != 'timestamp']
X_flat = flat_df[feature_cols].to_numpy()           # (N, n_features)
X = X_flat[:, np.newaxis, :]                        # (N, 1, n_features)
y = None                                             # unsupervised — no target
timestamps = flat_df['timestamp'].to_numpy()[:, np.newaxis] if 'timestamp' in flat_df.columns else None

print(f"X shape : {X.shape}")
print(f"y       : {y}")
print(f"Feature columns ({len(feature_cols)}): {feature_cols[:5]} ...")

## 3. Basic sanity checks on training data

In [ ]:
# Check for NaNs — IsolationForestHealth imputes them, but good to know upfront
X_2d = X.reshape(X.shape[0], -1)
nan_count = np.isnan(X_2d).sum()
nan_frac  = nan_count / X_2d.size
print(f"NaN count : {nan_count} ({nan_frac:.2%} of all values)")

# Per-feature NaN rate
nan_per_feat = np.isnan(X_2d).mean(axis=0)
high_nan = [(feature_cols[i], f"{nan_per_feat[i]:.1%}") for i in np.where(nan_per_feat > 0.05)[0]]
if high_nan:
    print("\nFeatures with >5% NaN:")
    for feat, pct in high_nan:
        print(f"  {feat}: {pct}")
else:
    print("No feature exceeds 5% NaN — clean data ✓")

# Variance check — zero-variance features confuse IsolationForest
var_per_feat = np.nanvar(X_2d, axis=0)
zero_var = [feature_cols[i] for i in np.where(var_per_feat == 0)[0]]
print(f"\nZero-variance features ({len(zero_var)}): {zero_var[:10]}")

## 4. Initialise Model and train

In [ ]:
# ── BUG NOTE: Model.__repr__ has a bug — it calls print() and returns None.
# The correct implementation should be:
#   def __repr__(self): return f"Model(name={self.model_name})"
# Workaround: just print model_name directly.

# Re-init handler with correct target name for unsupervised detection
handler = DataHandler(config=method_config, target_name="ced_machine__health_score")
handler.df = flat_df.copy()  # assign the flat training DataFrame

model = Model(
    data_handler=handler,
    model=method_config["method"],
    config=method_config,
    target_name="ced_machine__health_score",
)

print(f"Model name  : {model.model_name}")
print(f"Model type  : {model.model_type}")
print(f"Backend     : {type(model.backend).__name__}")
print(f"Unsupervised: {model.model_name in ['isolation_forest', 'var_forecast']}")

In [ ]:
# Train — expects X: (N, 1, F), y: None, timestamps optional
# model.train() will reshape to (N*1, F) = (N, F) via UNSUPERVISED_MODELS branch
model.train(X, y, timestamps)
print("\nTraining complete ✓")

## 5. Verify the saved model can be loaded back

In [ ]:
import joblib, os

save_path = f"saved_models/{method_config['save_name']}/model.joblib"
assert os.path.exists(save_path), f"Model file not found: {save_path}"

loaded_model = joblib.load(save_path)
print(f"Loaded model type: {type(loaded_model).__name__}")
print(f"IsolationForest n_estimators: {loaded_model.iso.n_estimators}")
print(f"Centroid shape  : {loaded_model.centroid_.shape}")
print(f"d_anchor_       : {loaded_model.d_anchor_:.4f}")
print(f"score_mean_     : {loaded_model.score_mean_:.4f}")
print(f"score_std_      : {loaded_model.score_std_:.4f}")

## 6. Batch predict health scores over the full dataset

In [ ]:
# Direct predict via the loaded IsolationForestHealth model
X_2d = X.reshape(X.shape[0], -1)   # (N, F) — same as what train() passed to .fit()
health_scores = loaded_model.predict(X_2d)

print(f"Health scores shape : {health_scores.shape}")
print(f"Score range         : {health_scores.min():.1f} – {health_scores.max():.1f}")
print(f"Mean health score   : {health_scores.mean():.1f}")
print(f"Std dev             : {health_scores.std():.1f}")

healthy  = (health_scores >= 75).sum()
degraded = ((health_scores >= 40) & (health_scores < 75)).sum()
critical = (health_scores < 40).sum()
print(f"\nHealthy  (≥75) : {healthy}  ({healthy/len(health_scores):.1%})")
print(f"Degraded (40-75): {degraded} ({degraded/len(health_scores):.1%})")
print(f"Critical (<40)  : {critical} ({critical/len(health_scores):.1%})")

## 7. Plot health scores over time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.axhspan(75, 100, alpha=0.07, color='green',  label='Healthy (≥75)')
ax.axhspan(40,  75, alpha=0.07, color='orange', label='Degraded (40–75)')
ax.axhspan( 0,  40, alpha=0.07, color='red',    label='Critical (<40)')

x_axis = pd.to_datetime(timestamps[:, -1]) if timestamps is not None else np.arange(len(health_scores))
ax.plot(x_axis, health_scores, color='#1565C0', linewidth=1.4, zorder=3)
ax.fill_between(x_axis, health_scores, alpha=0.12, color='#1565C0')

ax.axhline(75, color='green', linewidth=0.8, linestyle='--', alpha=0.6)
ax.axhline(40, color='red',   linewidth=0.8, linestyle='--', alpha=0.6)
ax.set_ylim(0, 105)
ax.set_title('CED Machine — Health Score over Time', fontsize=13, fontweight='bold')
ax.set_xlabel('Time')
ax.set_ylabel('Health Score (0–100)')
ax.legend(loc='lower left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('health_score_ced.png', dpi=120)
plt.show()
print("Plot saved to health_score_ced.png")

## 8. Per-feature contribution (anomaly score decomposition)

In [ ]:
# IsolationForest doesn't give per-feature importance natively,
# but we can look at per-feature deviation from centroid as a proxy.

from sklearn.preprocessing import StandardScaler

X_scaled = loaded_model.scaler.transform(
    loaded_model.imputer.transform(X_2d)
)

# Mean absolute deviation from training centroid, per feature
deviations = np.abs(X_scaled - loaded_model.centroid_).mean(axis=0)
feat_deviation = pd.Series(deviations, index=feature_cols).sort_values(ascending=False)

print("Top 10 features by mean deviation from healthy centroid:")
print(feat_deviation.head(10).to_string())

fig, ax = plt.subplots(figsize=(12, 4))
feat_deviation.head(20).plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Top 20 Features — Mean Deviation from Healthy Centroid')
ax.set_ylabel('Mean |deviation| (scaled units)')
ax.tick_params(axis='x', labelrotation=45)
plt.tight_layout()
plt.savefig('feature_deviation_ced.png', dpi=120)
plt.show()

## 9. Test `real_time_inference` on a single window

In [ ]:
# real_time_inference expects a DataFrame with shape (history_window, n_features+timestamp)
# For isolation_forest with history_window=1, this is a 1-row DataFrame

# Take the first window from flat_df as a test
single_window_df = flat_df[feature_cols].iloc[[0]].copy()
single_window_df['timestamp'] = pd.Timestamp.now()  # real_time_inference drops this

print("Input window shape:", single_window_df.shape)
print(single_window_df.head())

# ── BUG NOTE: real_time_inference checks `if model_type == 'sklearn' and method NOT in ROWWISE_MODELS`
# isolation_forest IS in ROWWISE_MODELS, so neither reshape runs.
# X after drop = (1, 72). sklearn model.predict() receives (1, 72) — this is correct for seq_len=1.
# If history_window were >1 you'd need to flatten manually here.

score = model.real_time_inference(single_window_df)
print(f"\nReal-time health score: {score}")
assert isinstance(score, list), "Expected list output"
assert 0 <= score[0] <= 100, f"Score out of range: {score[0]}"
print("✓ real_time_inference output is valid")

## 10. Stress test: real_time_inference over all windows

In [ ]:
rt_scores = []
errors = []

for i in range(len(flat_df)):
    window_df = flat_df[feature_cols].iloc[[i]].copy()
    window_df['timestamp'] = flat_df['timestamp'].iloc[i] if 'timestamp' in flat_df.columns else pd.Timestamp.now()
    try:
        s = model.real_time_inference(window_df)
        rt_scores.append(s[0])
    except Exception as e:
        errors.append((i, str(e)))
        rt_scores.append(np.nan)

rt_scores = np.array(rt_scores)
print(f"Processed {len(rt_scores)} windows")
print(f"Errors    : {len(errors)}")
if errors:
    print("First error:", errors[0])

print(f"\nRT score range : {np.nanmin(rt_scores):.1f} – {np.nanmax(rt_scores):.1f}")
print(f"RT mean score  : {np.nanmean(rt_scores):.1f}")

# Verify real_time_inference matches batch predict (should be identical)
max_diff = np.nanmax(np.abs(rt_scores - health_scores))
print(f"\nMax diff vs batch predict: {max_diff:.6f}")
if max_diff < 1e-4:
    print("✓ real_time_inference matches batch predict")
else:
    print("⚠ Mismatch between real_time and batch — investigate")

## 11. Score distribution histogram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(health_scores, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(75, color='green', linestyle='--', label='Healthy threshold (75)')
ax.axvline(40, color='red',   linestyle='--', label='Critical threshold (40)')
ax.set_xlabel('Health Score')
ax.set_ylabel('Count')
ax.set_title('CED Machine Health Score Distribution')
ax.legend()
plt.tight_layout()
plt.savefig('health_score_dist_ced.png', dpi=120)
plt.show()

## 12. Summary of bugs found

In [ ]:
bugs = [
    {
        'file': 'model.py',
        'location': 'Model.__repr__',
        'severity': 'Low (cosmetic)',
        'description': 'Uses print() instead of return. Returns None, not a string.',
        'fix': 'def __repr__(self): return f"Model(name={self.model_name})"'
    },
    {
        'file': 'model.py',
        'location': 'Model.real_time_inference — ROWWISE_MODELS branch',
        'severity': 'Medium (latent)',
        'description': (
            'isolation_forest is in ROWWISE_MODELS, so neither reshape branch fires. '
            'X is passed as (seq_len, F) raw. Works only because history_window=1 '
            'makes this equivalent to (1, F). If history_window > 1, predict() would '
            'receive wrong shape (seq_len, F) instead of (1, seq_len*F).'
        ),
        'fix': (
            'Add an explicit elif for ROWWISE_MODELS: '
            'X = X.reshape(1, -1)  # flatten seq_len*F for single-window inference'
        )
    },
    {
        'file': 'model.py / SklearnBackend.train',
        'location': 'UNSUPERVISED_MODELS branch in Model.train()',
        'severity': 'Low (note)',
        'description': (
            'isolation_forest is in both UNSUPERVISED_MODELS and ROWWISE_MODELS. '
            'The train() if/elif chain hits UNSUPERVISED_MODELS first (correct). '
            'ROWWISE_MODELS branch is never reached for isolation_forest during training, '
            'which is fine since the unsupervised reshape is identical.'
        ),
        'fix': 'No change needed, but add a comment clarifying the precedence.'
    },
    {
        'file': 'test.ipynb (original)',
        'location': 'Cell: handler.df = df',
        'severity': 'Medium',
        'description': (
            'Assigns flat_df back to handler.df after fetch_train_data() was already called. '
            'Then trains without re-calling fetch_train_data — so X, y, timestamps are from '
            'the original DataHandler run, but handler.df is now the flat CSV. '
            'Inconsistent state: handler and model are decoupled.'
        ),
        'fix': 'Either re-run fetch_train_data() after assigning handler.df, or skip the re-assignment.'
    },
]

print(f"{'='*80}")
print(f"BUGS FOUND: {len(bugs)}")
print(f"{'='*80}")
for i, b in enumerate(bugs, 1):
    print(f"\n[{i}] {b['file']} — {b['location']}")
    print(f"    Severity   : {b['severity']}")
    print(f"    Description: {b['description']}")
    print(f"    Fix        : {b['fix']}")